# M2.3a — Thống kê suy luận sensor đầu/cuối vòng đời trên FD001 train

Notebook này tạo artifact chính thức cho mốc M2.3a. Phân tích chỉ đọc `data/raw/train_FD001.txt`, tạo mean của 30 cycles đầu và 30 cycles cuối theo từng engine, rồi tính effect size, bootstrap confidence interval, Wilcoxon permutation và Benjamini–Hochberg FDR theo cấu hình M2.0 đã chốt.

## Phạm vi

- Đơn vị phân tích là train engine; mỗi engine đóng góp một difference `late_mean - early_mean` cho từng sensor.
- Notebook không đọc `test_FD001.txt` hoặc `RUL_FD001.txt`.
- Output của cụm này là CSV và JSON metadata/validation. Hình và diễn giải chi tiết sẽ thuộc cụm M2.3b.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data' / 'raw' / 'train_FD001.txt').exists():
            return candidate
    raise RuntimeError('Không tìm thấy thư mục gốc repository.')

PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
src_path = PROJECT_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
PROJECT_ROOT

WindowsPath('D:/IUH/BigData/Final_Project')

In [2]:
import pandas as pd

from cmapss.statistics import SensorInferenceConfig
from cmapss.statistics_artifacts import write_sensor_statistics_artifacts

## Cấu hình M2.0

Các tham số dưới đây được lấy từ `SensorInferenceConfig` và không phụ thuộc vào default ngầm của SciPy/NumPy.

In [3]:
config = SensorInferenceConfig()
config

SensorInferenceConfig(bootstrap_resamples=10000, bootstrap_confidence_level=0.95, bootstrap_method='percentile', bootstrap_quantile_method='linear', wilcoxon_alternative='two-sided', wilcoxon_zero_method='wilcox', wilcoxon_correction=False, wilcoxon_method='permutation', wilcoxon_permutation_resamples=100000, fdr_method='bh', fdr_q=0.05, fdr_family_size=21, seed=42)

## Tạo artifact

Hàm bên dưới đọc raw train, kiểm tra FD001 profile, tạo bảng cặp, tính bảng suy luận, ghi CSV bằng precision bảo toàn `float64`, đọc lại CSV và chạy validator round-trip trước khi ghi `run_info.json` và `validation.json`.

In [4]:
result = write_sensor_statistics_artifacts(
    project_root=PROJECT_ROOT,
    raw_dir='data/raw',
    output_dir='outputs/statistics',
    config=config,
)
result.validation

{'status': 'passed',
 'scope': 'M2.3a artifact validation for train-only sensor inference',
 'test_data_used': False,
 'test_labels_used': False,
 'raw_sha256_unchanged': True,
 'config_matches_m20': True,
 'engine_sensor_pairs': {'rows': 2100,
  'expected_rows': 2100,
  'roundtrip': 'passed',
  'pairs_per_sensor': 100,
  'sensors_per_engine': 21,
  'window_cycles': 30},
 'sensor_inference': {'rows': 21,
  'expected_rows': 21,
  'roundtrip': 'passed',
  'test_status_counts': {'all_zero_difference': 6, 'ok': 15},
  'bootstrap_status_counts': {'degenerate': 6, 'ok': 15},
  'fdr_reject_true': 15,
  'raw_p_values_defined': 15},
 'checks': ['read_observations(train) and validate_fd001_profile passed',
  'validate_engine_sensor_pairs passed before and after CSV round-trip',
  'validate_sensor_inference passed before and after CSV round-trip',
  'raw train SHA-256 unchanged',
  'bootstrap_resamples=10000',
  'wilcoxon_permutation_resamples=100000']}

## Kiểm tra nhanh bảng xuất ra

Bảng `sensor_inference.csv` giữ đủ 21 sensors. Notebook hiển thị các cột chính để kiểm tra hình dạng và trạng thái, chưa dùng bảng này để chọn feature.

In [5]:
display_columns = [
    'sensor', 'n_pairs', 'n_positive', 'n_negative', 'n_zero',
    'mean_difference', 'median_difference', 'ci_lower', 'ci_upper',
    'raw_p_value', 'p_value_adjusted', 'fdr_reject', 'test_status',
]
result.inference.loc[:, display_columns]

,sensor,n_pairs,n_positive,n_negative,n_zero,mean_difference,median_difference,ci_lower,ci_upper,raw_p_value,p_value_adjusted,fdr_reject,test_status
0,sensor_1,100,0,0,100,0.000000,0.000000,0.000000,0.000000,NaN,NaN,<NA>,all_zero_difference
1,sensor_2,100,100,0,0,0.999310,0.969000,0.959059,1.040183,0.00002,0.000028,True,ok
2,sensor_3,100,100,0,0,11.908667,11.681667,11.443951,12.377164,0.00002,0.000028,True,ok
3,sensor_4,100,100,0,0,20.128097,20.434333,19.402735,20.865606,0.00002,0.000028,True,ok
4,sensor_5,100,0,0,100,0.000000,0.000000,0.000000,0.000000,NaN,NaN,<NA>,all_zero_difference
5,sensor_6,100,47,0,53,0.000447,0.000000,0.000320,0.000580,0.00002,0.000028,True,ok
6,sensor_7,100,0,100,0,-1.913527,-1.886000,-1.999307,-1.829684,0.00002,0.000028,True,ok
7,sensor_8,100,100,0,0,0.130297,0.133333,0.119533,0.141294,0.00002,0.000028,True,ok
8,sensor_9,100,71,29,0,31.531620,17.630333,24.236336,39.319072,0.00002,0.000028,True,ok
9,sensor_10,100,0,0,100,0.000000,0.000000,0.000000,0.000000,NaN,NaN,<NA>,all_zero_difference


In [6]:
{
    'pair_rows': len(result.pairs),
    'inference_rows': len(result.inference),
    'output_dir': result.output_dir.as_posix(),
    'tables': result.run_info['tables'],
}

{'pair_rows': 2100,
 'inference_rows': 21,
 'output_dir': 'D:/IUH/BigData/Final_Project/outputs/statistics',
 'tables': ['outputs/statistics/tables/engine_sensor_pairs.csv',
  'outputs/statistics/tables/sensor_inference.csv']}